In [1]:
# BASE모델 , 파인튜닝 Instruct 성능 비교
# Qween2.5  Qween2.5_instruct
from transformers import AutoModelForCausalLM, AutoTokenizer

prompt = '프랑스 파리를 여행하려고 해 꼭 가봐야 할 명소를 두 곳만 추천해줘'
base_model_id = 'Qwen/Qwen2.5-0.5B'
instruct_model_id = 'Qwen/Qwen2.5-0.5B-Instruct'

def generate(model_id):
    model = AutoModelForCausalLM.from_pretrained(
        model_id,
        torch_dtype="auto",
        device_map="auto"
    )
    tokenizer = AutoTokenizer.from_pretrained(model_id)    
    messages = [
        {"role": "system", "content": "You are Qwen, created by Alibaba Cloud. You are a helpful assistant."},
        {"role": "user", "content": prompt}
    ]
    text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )
    model_inputs = tokenizer([text], return_tensors="pt").to(model.device)
    generated_ids = model.generate(
        **model_inputs,
        pad_token_id = tokenizer.eos_token_id,
        temperature = 0.1,
        do_sample=True,
        max_new_tokens=40
    )
    generated_ids = [
        output_ids[len(input_ids):] for input_ids, output_ids in zip(model_inputs.input_ids, generated_ids)
    ]
    response = tokenizer.batch_decode(generated_ids, skip_special_tokens=True)[0]
    return response

In [2]:
print(f'질문 : {prompt}')
base_output = generate(base_model_id)
print(f'[Base 모델 출력 : {base_output}]')
instruct_output = generate(instruct_model_id)
print(f'[Instruct 모델 출력 : {instruct_output}]')

질문 : 프랑스 파리를 여행하려고 해 꼭 가봐야 할 명소를 두 곳만 추천해줘


Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

[Base 모델 출력 : 프랑스 파리의 명소를 추천해드리겠습니다. 1. 파리의 유명한 명소 중 하나는 "파리의 유명한 명소 중 하나]


Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

[Instruct 모델 출력 : 프랑스의 파리에서 가장 인기 있는 명소 중 하나로는 "파리의 중심지인 로마"가 있습니다. 로마는 프랑스의 수도이며,]


In [3]:
# 정량적 평가 BLUE, ROUGE
from nltk.translate.bleu_score import sentence_bleu, SmoothingFunction
from rouge import Rouge
from kiwipiepy import Kiwi

Kiwi = Kiwi()  # 한국어 토큰화
# 한국어 형태소 단위 분리 함수
def tokenize_korean(text):
    return [token.form for token in Kiwi.tokenize(text)]

# 스무딩기법 점수가 불필요하게 0이되는 문제를 완화하기위해 사용
def evaluate_ngram(reference, candidate):
    smothie =  SmoothingFunction().method4
    ref_tokens = [tokenize_korean(reference)]
    cand_tokens = tokenize_korean(candidate)

    blue_score = sentence_bleu(ref_tokens,cand_tokens,smoothing_function=smothie)
    rouge = Rouge()
    ref_str = ' '.join(ref_tokens[0])
    cand_str = ' '.join(cand_tokens)
    scores = rouge.get_scores(cand_str,ref_str)[0]
    rouge_f1 = scores['rouge-1']['f']
    return blue_score,rouge_f1

ref = '파리의 명소로는 에펠탑과 루브르 박물관을 추천합니다.'
b_blue, b_rouge = evaluate_ngram(ref,base_output)
i_blue, i_rouge = evaluate_ngram(ref,instruct_output)

print(f'Base모델 BLUE : {b_blue} ROUGE-1 : {b_rouge}')
print(f'Instruct모델 BLUE : {i_blue} ROUGE-1 : {i_rouge}')

Base모델 BLUE : 0.055589753387051244 ROUGE-1 : 0.38709676932362125
Instruct모델 BLUE : 0.029398036975048066 ROUGE-1 : 0.285714281044898


In [4]:
# LM-Evaluation-Harness평가
# 모델의 추론 능력을 벤치마트 데이터셋(hellaswag)을 통해 accuracy를 측정
import lm_eval
def lm_eval_score(model_id):
    try:
        result = lm_eval.simple_evaluate(
            model="hf",
            model_args = f"pretrained={model_id},dtype=float32",
            tasks=['hellaswag'],
            device="cpu",
            limit=2
        )
        print(f"LM-Eval 정확도(acc): {result['results']['hellaswag']['acc,none']}")
    except Exception as e:
        print(e)
lm_eval_score(base_model_id)
print('='*100)
lm_eval_score(instruct_model_id)

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

Running loglikelihood requests: 100%|██████████| 8/8 [00:01<00:00,  5.59it/s]
pretrained=pretrained=Qwen/Qwen2.5-0.5B-Instruct,dtype=float32 appears to be an instruct or chat variant but chat template is not applied.
        Recommend setting `apply_chat_template` (optionally `fewshot_as_multiturn`).


LM-Eval 정확도(acc): 0.0


Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

Running loglikelihood requests: 100%|██████████| 8/8 [00:01<00:00,  5.60it/s]


LM-Eval 정확도(acc): 0.0


In [7]:
# GPT를 활용한 정성평가(LLM-as-a-Judge)
import os
from dotenv import load_dotenv
from openai import OpenAI
load_dotenv(override=True)
client = OpenAI()

In [10]:
def evaluate_with_gpt(generate_text):
    eval_prompt = f'''
질문:{prompt}
답변:{generate_text}
위 질문에 대한 답변의 정확성, 유창성을 1~5점으로 평가하고 이유를 100자이내로 작성하세요
'''
    try:
        response = client.chat.completions.create(
            model ='gpt-5.4-nano',
            messages=[{'role':'user','content':eval_prompt}],
            max_completion_tokens=150
        )
        return response.choices[0].message.content.strip()
    except Exception as e:
        return e
print('BASE model gpt eval')    
print(evaluate_with_gpt(base_output))
print('Instruct model gpt eval')    
print(evaluate_with_gpt(instruct_output))

BASE model gpt eval
평가: 1/5  
이유(100자 이내): 두 곳 추천이 아니라 문장이 반복되고 명소가 구체적으로 제시되지 않아 유창성과 정확성이 모두 부족함.
Instruct model gpt eval
- **정확성: 1/5**  
“파리의 중심지인 로마”는 실제 명소로도 존재하지 않으며, 로마가 아니라 파리의 대표명소를 잘못 답했습니다.  
- **유창성: 2/5**  
문장은 연결되지만 내용이 부정확해 전반적 흐름이 어색합니다.  

**총평(100자 이내 이유):**  
로마를 파리의 중심지 명소로 오답. 요청한 “꼭 가봐야 할 파리 명소 2곳”도 충족하지 못함.
